# Phase 10 — Power BI Export

**Purpose:** Package all V1 analysis outputs into a clean, dashboard-ready data model for Power BI.

**Inputs:**
- `data/clean/clean_master_sa2_v2.csv` — 176 SA2s with dual tiers, corrected billing, hardship estimates
- `outputs/simulation/scenario_results.csv` — 1408 rows (176 SA2s × 8 scenarios)
- `data/clean/shap_values_critical.csv` — 168 SA2s with SHAP values for 6 SEIFA features

**Outputs** (all to `powerbi/`):
- `sa2_main.csv` — 176 rows, SA2 dimension + affordability + both tier systems + SHAP top driver
- `simulation_scenarios.csv` — 1408 rows, scenario fact table
- `shap_long.csv` — 1008 rows (168 SA2s × 6 features), long format for bar charts

**Join key:** `SA2_CODE21` (string — leading zeros preserved in all tables)

**Tier systems:**
- `burden_tier_rel` — percentile-based relative ranking (Critical/High/Moderate/Low) — used for ML and prioritisation
- `burden_tier_abs` — absolute policy thresholds (>4%/3-4%/2-3%/<2%) — used for simulation and policy

**Power BI helpers added:**
- `tier_order_rel` / `tier_order_abs` (1=Critical → 4=Low) — correct tier sort order in visuals
- `*_pct` columns — ratio × 100 for human-readable display
- `scenario_label` — e.g. 'Baseline', '+5%', 'FY2025-26 (exact)' — for scenario slicer
- `tier_shifted` — boolean flag where SA2 changed absolute tier vs baseline

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("..")
CLEAN = ROOT / "data" / "clean"
SIM   = ROOT / "outputs" / "simulation"
OUT   = ROOT / "powerbi"
OUT.mkdir(exist_ok=True)

## 1. Load inputs

In [2]:
v2   = pd.read_csv(CLEAN / "clean_master_sa2_v2.csv",  dtype={"SA2_CODE21": str})
sim  = pd.read_csv(SIM   / "scenario_results.csv",     dtype={"SA2_CODE21": str})
shap = pd.read_csv(CLEAN / "shap_values_critical.csv", dtype={"SA2_CODE21": str})

print(f"v2   : {v2.shape}")
print(f"sim  : {sim.shape}")
print(f"shap : {shap.shape}")
print()
print("v2 columns:", v2.columns.tolist())
print()
print("sim columns:", sim.columns.tolist())
print()
print("shap columns:", shap.columns.tolist())

v2   : (176, 37)
sim  : (1408, 18)
shap : (168, 18)

v2 columns: ['SA2_CODE21', 'SA2_NAME21', 'SA3_NAME21', 'SA4_NAME21', 'population', 'AREASQKM21', 'provider_type', 'provider_note', 'is_provider_confirmed', 'is_metro', 'irsd_score', 'irsd_decile', 'irsad_score', 'irsad_decile', 'ier_score', 'ier_decile', 'ieo_score', 'ieo_decile', 'median_hhd_inc_weekly_2021', 'median_hhd_inc_annual_2021', 'median_hhd_inc_annual_adj', 'bill_usage_only', 'bill_usage_supply', 'bill_owner_metro', 'bill_owner_country', 'bill_owner_blended', 'water_cost_burden_ratio', 'burden_ratio_renter', 'burden_ratio_owner_metro', 'burden_ratio_owner_country', 'burden_tier_abs', 'burden_tier_rel', 'estimated_hardship_need', 'hardship_priority_score', 'water_stress_index', 'stress_pct_rank', 'stress_tier']

sim columns: ['price_increase_pct', 'scenario_label', 'SA2_CODE21', 'SA2_NAME21', 'SA3_NAME21', 'SA4_NAME21', 'provider_type', 'is_metro', 'burden_tier_abs', 'burden_tier_rel', 'median_hhd_inc_annual_adj', 'sim_annu

## 2. Helper — tier order mapping

Power BI sorts text alphabetically by default. Adding `tier_order` allows correct
sort order (Critical first) on any visual that uses a burden tier as an axis.

In [3]:
TIER_ORDER = {"Critical": 1, "High": 2, "Moderate": 3, "Low": 4, "Unknown": 5}

def add_tier_order(df, tier_col, order_col):
    df = df.copy()
    df[order_col] = df[tier_col].map(TIER_ORDER)
    return df

## 3. Build `sa2_main.csv`

Full SA2 dimension table. Merges SHAP top driver (left join so all 176 SA2s are
kept; non-SA Water and Unknown-tier SA2s will have NaN for SHAP columns).

In [4]:
# Bring in SHAP top driver only (avoid duplicating SHAP value cols in main table)
shap_join = shap[["SA2_CODE21", "top_critical_driver", "top_critical_driver_shap"]].copy()

sa2_main = v2.merge(shap_join, on="SA2_CODE21", how="left")

print(f"After merge: {sa2_main.shape}")
assert sa2_main.shape[0] == 176, "Row count changed — check SA2_CODE21 join"

After merge: (176, 39)


In [5]:
# Percentage columns for display (ratio → %)
sa2_main["water_cost_burden_pct"]         = (sa2_main["water_cost_burden_ratio"] * 100).round(2)
sa2_main["burden_ratio_owner_metro_pct"]  = (sa2_main["burden_ratio_owner_metro"] * 100).round(2)
sa2_main["burden_ratio_owner_country_pct"]= (sa2_main["burden_ratio_owner_country"] * 100).round(2)
sa2_main["burden_ratio_renter_pct"]       = (sa2_main["burden_ratio_renter"] * 100).round(2)
sa2_main["hardship_priority_pct"]         = (sa2_main["hardship_priority_score"] * 100).round(2)

# Tier sort orders for both systems
sa2_main = add_tier_order(sa2_main, "burden_tier_rel", "tier_order_rel")
sa2_main = add_tier_order(sa2_main, "burden_tier_abs", "tier_order_abs")

# Hardship priority flag — SA Water SA2s in top relative tiers
sa2_main["hardship_priority_flag"] = (
    (sa2_main["provider_type"] == "SA Water") &
    sa2_main["burden_tier_rel"].isin(["Critical", "High"])
).astype(int)

# Boolean → int for Power BI compatibility
sa2_main["is_metro"]              = sa2_main["is_metro"].astype(int)
sa2_main["is_provider_confirmed"] = sa2_main["is_provider_confirmed"].astype(int)

print(sa2_main.shape)
print(sa2_main.dtypes)
sa2_main.head(3)

(176, 47)
SA2_CODE21                            str
SA2_NAME21                            str
SA3_NAME21                            str
SA4_NAME21                            str
population                        float64
AREASQKM21                        float64
provider_type                         str
provider_note                         str
is_provider_confirmed               int64
is_metro                            int64
irsd_score                        float64
irsd_decile                       float64
irsad_score                       float64
irsad_decile                      float64
ier_score                         float64
ier_decile                        float64
ieo_score                         float64
ieo_decile                        float64
median_hhd_inc_weekly_2021          int64
median_hhd_inc_annual_2021          int64
median_hhd_inc_annual_adj         float64
bill_usage_only                   float64
bill_usage_supply                 float64
bill_owner_metro        

,SA2_CODE21,SA2_NAME21,SA3_NAME21,SA4_NAME21,population,AREASQKM21,provider_type,provider_note,is_provider_confirmed,is_metro,...,top_critical_driver,top_critical_driver_shap,water_cost_burden_pct,burden_ratio_owner_metro_pct,burden_ratio_owner_country_pct,burden_ratio_renter_pct,hardship_priority_pct,tier_order_rel,tier_order_abs,hardship_priority_flag
0,401011001,Adelaide,Adelaide City,Adelaide - Central and Hills,18202.0,10.4824,SA Water,NaN,0,1,...,IRSAD (Adv/Disadv),-0.088112,1.43,1.43,1.66,0.97,145.17,3,4,0
1,401011002,North Adelaide,Adelaide City,Adelaide - Central and Hills,6823.0,5.0909,SA Water,NaN,0,1,...,IRSAD (Adv/Disadv),-0.073686,1.05,1.05,1.21,0.71,45.45,4,4,0
2,401021003,Adelaide Hills,Adelaide Hills,Adelaide - Central and Hills,7051.0,364.4390,SA Water,NaN,0,1,...,IRSAD (Adv/Disadv),-0.068744,1.03,1.03,1.19,0.70,44.14,4,4,0


In [6]:
print("=== Tier distributions ===")
print("\nburden_tier_rel:")
print(sa2_main.groupby(["tier_order_rel", "burden_tier_rel"]).size().reset_index(name="count"))
print("\nburden_tier_abs:")
print(sa2_main.groupby(["tier_order_abs", "burden_tier_abs"]).size().reset_index(name="count"))
print("\nprovider_type:")
print(sa2_main["provider_type"].value_counts())

=== Tier distributions ===

burden_tier_rel:
   tier_order_rel burden_tier_rel  count
0               1        Critical     17
1               2            High     25
2               3        Moderate     84
3               4             Low     42
4               5         Unknown      8

burden_tier_abs:
   tier_order_abs burden_tier_abs  count
0               3        Moderate     24
1               4             Low    144
2               5         Unknown      8

provider_type:
provider_type
SA Water    175
Council       1
Name: count, dtype: int64


## 4. Build `simulation_scenarios.csv`

Long-format scenario fact table. One row per SA2 per price scenario.
Joined to `sa2_main` in Power BI via `SA2_CODE21`.

Scenarios: Baseline, +5%, +10%, +15%, +20%, +25%, +30%, FY2025-26 (exact)

In [7]:
# scenario_label is already set in Phase 8 output
print("Scenarios in file:")
print(sim[["price_increase_pct", "scenario_label"]].drop_duplicates().sort_values("price_increase_pct"))

Scenarios in file:
      price_increase_pct     scenario_label
0                    0.0           Baseline
1232                 4.7  FY2025-26 (exact)
176                  5.0                +5%
352                 10.0               +10%
528                 15.0               +15%
704                 20.0               +20%
880                 25.0               +25%
1056                30.0               +30%


In [8]:
# Bring baseline absolute tier from sa2_main for comparison
baseline_lookup = sa2_main[["SA2_CODE21", "burden_tier_abs", "burden_tier_rel"]].rename(
    columns={"burden_tier_abs": "baseline_tier_abs", "burden_tier_rel": "baseline_tier_rel"}
)
sim_out = sim.merge(baseline_lookup, on="SA2_CODE21", how="left")

# Tier order for baseline absolute tier
sim_out = add_tier_order(sim_out, "baseline_tier_abs", "baseline_tier_order")

# sim_tier_order was computed in Phase 8 against sim_tier_abs
# Confirm tier_shifted aligns with sim_tier_order < baseline_tier_order
check = (sim_out["sim_tier_order"] < sim_out["baseline_tier_order"]).astype(int)
mismatch = (check != sim_out["tier_shifted"]).sum()
print(f"tier_shifted consistency check — mismatches: {mismatch}")

# Keep logical column order
sim_out = sim_out[[
    "SA2_CODE21", "SA2_NAME21", "SA3_NAME21", "SA4_NAME21",
    "provider_type", "is_metro",
    "price_increase_pct", "scenario_label",
    "baseline_tier_abs", "baseline_tier_rel", "baseline_tier_order",
    "burden_tier_abs", "burden_tier_rel",
    "median_hhd_inc_annual_adj",
    "sim_annual_bill", "sim_burden_ratio", "sim_burden_pct",
    "sim_tier_abs", "sim_tier_order",
    "tier_shift", "tier_shifted"
]]

# Boolean → int for Power BI compatibility
sim_out["is_metro"] = sim_out["is_metro"].astype(int)

print(f"\nsim_out shape: {sim_out.shape}")
print(sim_out.dtypes)

tier_shifted consistency check — mismatches: 0

sim_out shape: (1408, 21)
SA2_CODE21                       str
SA2_NAME21                       str
SA3_NAME21                       str
SA4_NAME21                       str
provider_type                    str
is_metro                       int64
price_increase_pct           float64
scenario_label                   str
baseline_tier_abs                str
baseline_tier_rel                str
baseline_tier_order            int64
burden_tier_abs                  str
burden_tier_rel                  str
median_hhd_inc_annual_adj    float64
sim_annual_bill              float64
sim_burden_ratio             float64
sim_burden_pct               float64
sim_tier_abs                     str
sim_tier_order                 int64
tier_shift                   float64
tier_shifted                   int64
dtype: object


In [9]:
# Sanity: baseline scenario burden_tier_abs should match sa2_main
baseline_sim = sim_out[sim_out["scenario_label"] == "Baseline"][["SA2_CODE21", "sim_tier_abs"]]
check_merge = baseline_sim.merge(sa2_main[["SA2_CODE21", "burden_tier_abs"]], on="SA2_CODE21")
mismatches = check_merge[check_merge["sim_tier_abs"] != check_merge["burden_tier_abs"]]
print(f"Baseline absolute tier mismatches vs sa2_main: {len(mismatches)}")
if len(mismatches) > 0:
    print(mismatches)

print("\nTier shifts by scenario:")
print(sim_out[sim_out["tier_shifted"] == 1].groupby("scenario_label")["SA2_CODE21"].count().sort_values(ascending=False))

Baseline absolute tier mismatches vs sa2_main: 0

Tier shifts by scenario:


scenario_label
+30%                 17
+25%                 13
+20%                 11
+15%                  8
+10%                  5
FY2025-26 (exact)     5
+5%                   4
Name: SA2_CODE21, dtype: int64


## 5. Build `shap_long.csv`

Melt SHAP feature columns to long format: one row per SA2 per feature.
Enables easy bar charts in Power BI (feature on axis, SHAP value as bar).

Features: 6 SEIFA proxies only (no income/bill — proxy-risk model, not leaky model).

In [10]:
SHAP_COLS = [
    "shap_irsd_score",
    "shap_irsad_score",
    "shap_ier_score",
    "shap_ieo_score",
    "shap_population",
    "shap_AREASQKM21",
]

FEATURE_LABEL = {
    "shap_irsd_score":   "IRSD (Disadvantage)",
    "shap_irsad_score":  "IRSAD (Adv & Disadv)",
    "shap_ier_score":    "IER (Econ Resources)",
    "shap_ieo_score":    "IEO (Education & Occ)",
    "shap_population":   "Population",
    "shap_AREASQKM21":   "Area (km²)",
}

# Validate all SHAP cols present
missing = [c for c in SHAP_COLS if c not in shap.columns]
if missing:
    raise ValueError(f"Missing SHAP columns: {missing}")
print(f"SHAP cols present: {len(SHAP_COLS)} / {len(SHAP_COLS)}")

shap_id_cols = ["SA2_CODE21", "SA2_NAME21", "burden_tier_rel", "burden_tier_abs"]

shap_long = shap[shap_id_cols + SHAP_COLS].melt(
    id_vars=shap_id_cols,
    var_name="shap_feature",
    value_name="shap_value"
)

shap_long["feature_label"]  = shap_long["shap_feature"].map(FEATURE_LABEL)
shap_long["abs_shap_value"] = shap_long["shap_value"].abs()
shap_long = add_tier_order(shap_long, "burden_tier_rel", "tier_order_rel")
shap_long = add_tier_order(shap_long, "burden_tier_abs", "tier_order_abs")

n_sa2 = shap["SA2_CODE21"].nunique()
expected = n_sa2 * len(SHAP_COLS)
print(f"shap_long shape: {shap_long.shape}  (expected {n_sa2} SA2s × {len(SHAP_COLS)} features = {expected} rows)")
assert shap_long.shape[0] == expected, f"Row count mismatch: {shap_long.shape[0]} vs {expected}"
print(shap_long.dtypes)
shap_long.head(5)

SHAP cols present: 6 / 6
shap_long shape: (1008, 10)  (expected 168 SA2s × 6 features = 1008 rows)
SA2_CODE21             str
SA2_NAME21             str
burden_tier_rel        str
burden_tier_abs        str
shap_feature           str
shap_value         float64
feature_label          str
abs_shap_value     float64
tier_order_rel       int64
tier_order_abs       int64
dtype: object


,SA2_CODE21,SA2_NAME21,burden_tier_rel,burden_tier_abs,shap_feature,shap_value,feature_label,abs_shap_value,tier_order_rel,tier_order_abs
0,401011001,Adelaide,Moderate,Low,shap_irsd_score,-0.009794,IRSD (Disadvantage),0.009794,3,4
1,401011002,North Adelaide,Low,Low,shap_irsd_score,-0.065721,IRSD (Disadvantage),0.065721,4,4
2,401021003,Adelaide Hills,Low,Low,shap_irsd_score,-0.060361,IRSD (Disadvantage),0.060361,4,4
3,401021004,Aldgate - Stirling,Low,Low,shap_irsd_score,-0.051300,IRSD (Disadvantage),0.051300,4,4
4,401021005,Hahndorf - Echunga,Low,Low,shap_irsd_score,-0.065010,IRSD (Disadvantage),0.065010,4,4


## 6. Save to `powerbi/`

In [11]:
sa2_main.to_csv(OUT / "sa2_main.csv",             index=False)
sim_out.to_csv( OUT / "simulation_scenarios.csv", index=False)
shap_long.to_csv(OUT / "shap_long.csv",           index=False)

print("Saved:")
for f in sorted(OUT.iterdir()):
    rows = pd.read_csv(f).shape[0]
    print(f"  powerbi/{f.name}  ({rows:,} rows)")

Saved:


  powerbi/sa2_main.csv  (176 rows)


  powerbi/shap_long.csv  (1,008 rows)


  powerbi/simulation_scenarios.csv  (1,408 rows)


## 7. Data model summary

Table relationships for Power BI:

```
sa2_main (176 rows)  ←──[SA2_CODE21]──→  simulation_scenarios (1408 rows)
sa2_main (176 rows)  ←──[SA2_CODE21]──→  shap_long (1008 rows)
```

Both `simulation_scenarios` and `shap_long` are many-side tables; `sa2_main` is the one-side dimension.

**Dual tier fields:**
- `burden_tier_rel` / `tier_order_rel` — relative percentile tiers (ML target, suburb prioritisation)
- `burden_tier_abs` / `tier_order_abs` — absolute policy thresholds (simulation, policy reporting)
- For Power BI, use `burden_tier_rel` for the vulnerability map and hardship priority visuals;
  use `burden_tier_abs` + `sim_tier_abs` for the simulation / tipping-point pages.

**Recommended Power BI pages:**
1. **Suburb Vulnerability Map** — choropleth of `burden_tier_rel` / `water_cost_burden_pct`, slicer by SA4
2. **Price Rise Simulator** — slicer on `scenario_label`, map colours update by `sim_tier_abs`, card showing count of `tier_shifted`
3. **Tipping Point Table** — SA2s where `tier_shifted = 1`, ordered by lowest `price_increase_pct` at first shift
4. **Hardship Priority** — scatter: `water_cost_burden_pct` vs `estimated_hardship_need`, colour by `burden_tier_rel`, filter provider_type = SA Water
5. **SHAP Explainability** — bar chart from `shap_long`: `feature_label` on axis, `abs_shap_value` as bar, filter to Critical tier (use `tier_order_rel = 1`)

In [12]:
print("=== Final output summary ===")
print(f"sa2_main             : {sa2_main.shape[0]} rows × {sa2_main.shape[1]} cols")
print(f"simulation_scenarios : {sim_out.shape[0]} rows × {sim_out.shape[1]} cols")
print(f"shap_long            : {shap_long.shape[0]} rows × {shap_long.shape[1]} cols")
print()
print("=== Tier distribution — relative (sa2_main) ===")
print(sa2_main.groupby(["tier_order_rel", "burden_tier_rel"]).size().reset_index(name="count"))
print()
print("=== Tier distribution — absolute (sa2_main) ===")
print(sa2_main.groupby(["tier_order_abs", "burden_tier_abs"]).size().reset_index(name="count"))
print()
print("=== Absolute tier shifts by scenario ===")
shifts = sim_out[sim_out["tier_shifted"] == 1].groupby("scenario_label")["SA2_CODE21"].count()
print(shifts.sort_values(ascending=False))
print()
print("=== Provider types ===")
print(sa2_main["provider_type"].value_counts())
print()
print("=== Hardship priority SA2s (relative Critical/High, SA Water only) ===")
priority = sa2_main[sa2_main["hardship_priority_flag"] == 1][["SA2_NAME21", "burden_tier_rel", "water_cost_burden_pct", "estimated_hardship_need"]].sort_values("water_cost_burden_pct", ascending=False)
print(f"Count: {len(priority)}")
print(priority.head(10))

=== Final output summary ===
sa2_main             : 176 rows × 47 cols
simulation_scenarios : 1408 rows × 21 cols
shap_long            : 1008 rows × 10 cols

=== Tier distribution — relative (sa2_main) ===
   tier_order_rel burden_tier_rel  count
0               1        Critical     17
1               2            High     25
2               3        Moderate     84
3               4             Low     42
4               5         Unknown      8

=== Tier distribution — absolute (sa2_main) ===
   tier_order_abs burden_tier_abs  count
0               3        Moderate     24
1               4             Low    144
2               5         Unknown      8

=== Absolute tier shifts by scenario ===
scenario_label
+30%                 17
+25%                 13
+20%                 11
+15%                  8
+10%                  5
FY2025-26 (exact)     5
+5%                   4
Name: SA2_CODE21, dtype: int64

=== Provider types ===
provider_type
SA Water    175
Council       1
Name: cou